# LoRA Fine-Tuning of CLMBR for Three-Year Lung Cancer Risk

Task-specific parameter-efficient fine-tuning of `StanfordShahLab/gpt-base-2048-clmbr` on the leakage-safe MEDS timelines created by notebook 04.

This is a distinct experiment from notebook 05A: notebook 05A freezes CLMBR and trains a downstream model on fixed embeddings; this notebook updates low-rank adapters inside CLMBR plus a binary classification head. Each outer fold starts from the original pretrained checkpoint, uses no test-fold labels during training or model selection, and produces out-of-fold predictions for notebook 06. After the five-fold run, a downstream section reuses each saved fold adapter to extract fine-tuned representations and combines them with demographics in XGBoost without re-fine-tuning CLMBR.

Run this notebook on a GPU app. Completed LoRA folds are reused only when their saved training configuration matches the current settings; the demographic + fine-tuned-embedding section reuses those adapters and the existing 512-token cache.

## 1. Dependencies and model access

Run once per GPU app, then restart the kernel if packages were newly installed. The CLMBR checkpoint is gated: first accept its access conditions on Hugging Face, then provide an approved token through `HF_TOKEN` or `huggingface-cli login`.


In [ ]:
# Pin one tested Hugging Face stack without replacing Workbench's CUDA Torch.
# Install transitive dependencies for this stack; install hf-ehr itself with
# --no-deps because its training-only dependency set would downgrade the
# notebook's scientific Python environment.
%pip install -q --upgrade "huggingface-hub==0.25.2" "transformers==4.45.2" "tokenizers==0.20.1" "peft==0.13.2" "accelerate==1.0.1" "omegaconf==2.3.0" "loguru==0.7.2"
%pip install -q --upgrade --no-deps "hf-ehr==0.1.4"


## 2. Configuration and cohort

In [ ]:
from pathlib import Path
import gc
import json
import math
import os
import random
import shutil
import subprocess
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
MODEL_NAME = "StanfordShahLab/gpt-base-2048-clmbr"
MAX_LENGTH = 512       # A100 pilot setting; pretrained maximum is 2048.
BATCH_SIZE = 16         # A100 default; reduce to 8 only after a CUDA OOM.
GRAD_ACCUM_STEPS = 2    # Effective batch size remains 32.
EPOCHS = 10            # Maximum; best validation-AUPRC epoch is retained.
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
VALIDATION_FRACTION = 0.10
NUM_WORKERS = 2
FOLDS_TO_RUN = [0, 1, 2, 3, 4]
# Normally leave empty. Saved folds are reused only when their recorded
# configuration matches the signature below.
FORCE_RETRAIN_FOLDS = []
SYNTHETIC_MODE = os.environ.get(
    "LUNG_CANCER_SYNTHETIC", "0"
).strip().lower() in {"1", "true", "yes"}

EXPECTED_TRAINING_CONFIG = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "max_epochs": EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "validation_fraction": VALIDATION_FRACTION,
    "seed": SEED,
}

DATA_DIR = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
RUN_DIR = DATA_DIR / "paper_reproduction"
PRED_DIR = RUN_DIR / "predictions"
LORA_DIR = RUN_DIR / "clmbr_lora"
CACHE_DIR = DATA_DIR / f"clmbr_token_cache_{MAX_LENGTH}"
COHORT_PATH = DATA_DIR / "demo_baseline_paper_reproduction.parquet"
MEDS_PATH = DATA_DIR / "EHR_MEDS.parquet"
FOLD_PATH = RUN_DIR / "cv_folds.parquet"
for directory in [RUN_DIR, PRED_DIR, LORA_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BUCKET_RESOURCE_ID = os.environ.get(
    "LUNG_CANCER_BUCKET_RESOURCE_ID", ""
).strip()


def resolve_workspace_bucket():
    legacy_bucket = os.environ.get("WORKSPACE_BUCKET", "").strip()
    if legacy_bucket:
        return legacy_bucket.rstrip("/")
    if not shutil.which("wb"):
        raise RuntimeError(
            "The Workbench `wb` CLI is required to recover controlled CLMBR inputs."
        )
    if not BUCKET_RESOURCE_ID:
        listing = subprocess.run(
            ["wb", "resource", "list"],
            check=True, capture_output=True, text=True,
        ).stdout
        raise RuntimeError(
            "Set LUNG_CANCER_BUCKET_RESOURCE_ID to a controlled GCS bucket "
            "resource ID from `wb resource list`.\n\nCurrent resources:\n" + listing
        )
    output = subprocess.run(
        [
            "wb", "resource", "resolve",
            f"--id={BUCKET_RESOURCE_ID}", "--format=TEXT",
        ],
        check=True, capture_output=True, text=True,
    ).stdout
    lines_out = [line.strip() for line in output.splitlines() if line.strip()]
    if not lines_out:
        raise RuntimeError(
            f"Workbench returned no bucket path for resource {BUCKET_RESOURCE_ID!r}."
        )
    bucket = lines_out[-1].strip("`'\"").rstrip("/")
    return bucket if bucket.startswith("gs://") else "gs://" + bucket


# Cloud apps have separate local disks. Recover controlled inputs from the
# workspace bucket unless this is an explicitly marked synthetic smoke test.
if SYNTHETIC_MODE:
    print("Synthetic mode: using local cohort and MEDS fixtures; skipping bucket recovery.")
else:
    BUCKET = resolve_workspace_bucket()

    for local_path, object_name in [
        (COHORT_PATH, "demo_baseline_paper_reproduction.parquet"),
        (MEDS_PATH, "EHR_MEDS.parquet"),
    ]:
        if not local_path.exists():
            print("Downloading controlled input:", object_name)
            subprocess.run([
                "wb", "gcloud", "storage", "cp",
                f"{BUCKET}/clmbr/{object_name}", str(local_path),
            ], check=True)

for required in [COHORT_PATH, MEDS_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Missing {required} after bucket recovery.")
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for LoRA fine-tuning.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

cohort = pd.read_parquet(COHORT_PATH).reset_index(drop=True)
cohort["person_id"] = pd.to_numeric(cohort["person_id"], errors="raise").astype("int64")
cohort["has_lungcancer_3Y"] = pd.to_numeric(
    cohort["has_lungcancer_3Y"], errors="raise"
).astype("int8")
if not FOLD_PATH.exists():
    expected = cohort[["person_id", "has_lungcancer_3Y"]].copy()
    fold_values = np.full(len(expected), -1, dtype=int)
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold_id, (_, test_index) in enumerate(
        splitter.split(expected, expected["has_lungcancer_3Y"])
    ):
        fold_values[test_index] = fold_id
    pd.DataFrame({
        "person_id": expected["person_id"], "fold": fold_values,
    }).to_parquet(FOLD_PATH, index=False)
    print("Recreated deterministic shared folds:", FOLD_PATH)

folds = pd.read_parquet(FOLD_PATH)
folds["person_id"] = pd.to_numeric(folds["person_id"], errors="raise").astype("int64")
cohort = cohort.merge(folds[["person_id", "fold"]], on="person_id", how="left", validate="one_to_one")
if cohort["fold"].isna().any():
    raise ValueError("Saved folds do not cover the current cohort.")
cohort["fold"] = cohort["fold"].astype(int)
print("GPU:", torch.cuda.get_device_name(0))
print("Mixed precision:", "BF16" if USE_BF16 else "FP16")
print("Cohort:", cohort.shape, "cases:", int(cohort.has_lungcancer_3Y.sum()))
display(cohort.groupby("fold")["has_lungcancer_3Y"].agg(["count", "sum"]))

## 3. Build a reusable token cache

The cache contains only events already truncated before each participant's lifestyle-survey index date by notebook 04. It stores the most recent `MAX_LENGTH` tokens and contains no outcome labels.

This cell is resumable at the notebook level: if the completed cache metadata exists and matches the cohort and context length, it is loaded instead of rebuilt.

In [ ]:
import polars as pl
from hf_ehr.config import Event
from hf_ehr.data.tokenization import CLMBRTokenizer

TOKEN_IDS_PATH = CACHE_DIR / "input_ids.int32.mmap"
TOKEN_LENGTHS_PATH = CACHE_DIR / "lengths.int16.npy"
TOKEN_META_PATH = CACHE_DIR / "metadata.json"

tokenizer = CLMBRTokenizer.from_pretrained(MODEL_NAME)
pad_token_id = getattr(tokenizer, "pad_token_id", None)
if pad_token_id is None:
    pad_token_id = 0

expected_meta = {
    "model_name": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "n_participants": len(cohort),
    "person_id_checksum": int(cohort.person_id.sum()),
}

cache_ok = False
if TOKEN_META_PATH.exists() and TOKEN_IDS_PATH.exists() and TOKEN_LENGTHS_PATH.exists():
    saved_meta = json.loads(TOKEN_META_PATH.read_text())
    cache_ok = all(saved_meta.get(k) == v for k, v in expected_meta.items())

if not cache_ok:
    print("Building token cache from", MEDS_PATH)
    token_ids = np.memmap(
        TOKEN_IDS_PATH, mode="w+", dtype=np.int32, shape=(len(cohort), MAX_LENGTH)
    )
    token_ids[:] = int(pad_token_id)
    lengths = np.zeros(len(cohort), dtype=np.int16)
    row_for_id = pd.Series(cohort.index.to_numpy(), index=cohort.person_id).to_dict()

    # MEDS_FINAL was sorted by subject and time in notebook 04. Group once here;
    # retain only the most recent events needed by this experiment.
    grouped = (
        pl.scan_parquet(MEDS_PATH)
        .select(["subject_id", "time", "code"])
        .sort(["subject_id", "time"])
        .group_by("subject_id", maintain_order=True)
        .agg(pl.col("code").tail(MAX_LENGTH - 2).alias("codes"))
        .collect(streaming=True)
    )

    for person_id, codes in tqdm(grouped.iter_rows(), total=grouped.height, desc="Tokenizing patients"):
        row = row_for_id.get(int(person_id))
        if row is None or codes is None:
            continue
        events = [
            Event(code="SNOMED/184099003" if value == "MEDS_BIRTH" else value)
            for value in codes
        ]
        if len(events) < 5:
            continue
        encoded = tokenizer(
            [events], add_special_tokens=True, padding=False, return_tensors="pt"
        )["input_ids"][0].cpu().numpy().astype(np.int32)
        encoded = encoded[-MAX_LENGTH:]
        token_ids[row, : len(encoded)] = encoded
        lengths[row] = len(encoded)

    token_ids.flush()
    np.save(TOKEN_LENGTHS_PATH, lengths)
    meta = expected_meta | {
        "pad_token_id": int(pad_token_id),
        "participants_with_tokens": int((lengths >= 5).sum()),
    }
    TOKEN_META_PATH.write_text(json.dumps(meta, indent=2))
else:
    lengths = np.load(TOKEN_LENGTHS_PATH)
    print("Using existing token cache")

token_ids = np.memmap(
    TOKEN_IDS_PATH, mode="r", dtype=np.int32, shape=(len(cohort), MAX_LENGTH)
)
lengths = np.load(TOKEN_LENGTHS_PATH)
eligible_for_lora = lengths >= 5
print("Participants with trainable timelines:", int(eligible_for_lora.sum()))
print("Missing/short timelines:", int((~eligible_for_lora).sum()))
print("Median tokens:", int(np.median(lengths[eligible_for_lora])))

## 4. Dataset, dynamic padding, and LoRA classifier

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM


class CachedTimelineDataset(Dataset):
    def __init__(self, rows):
        self.rows = np.asarray(rows, dtype=np.int64)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, item):
        row = int(self.rows[item])
        length = int(lengths[row])
        return {
            "row": row,
            "input_ids": torch.as_tensor(np.asarray(token_ids[row, :length], dtype=np.int64)),
            "label": float(cohort.iloc[row].has_lungcancer_3Y),
        }


def collate_batch(items):
    batch_length = max(len(item["input_ids"]) for item in items)
    ids = torch.full((len(items), batch_length), int(pad_token_id), dtype=torch.long)
    mask = torch.zeros((len(items), batch_length), dtype=torch.long)
    labels = torch.tensor([item["label"] for item in items], dtype=torch.float32)
    rows = torch.tensor([item["row"] for item in items], dtype=torch.long)
    for i, item in enumerate(items):
        n = len(item["input_ids"])
        ids[i, :n] = item["input_ids"]
        mask[i, :n] = 1
    return {"input_ids": ids, "attention_mask": mask, "labels": labels, "rows": rows}


class CLMBRLoRAClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        causal_lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
        hidden_size = int(causal_lm.config.n_embd)
        encoder = causal_lm.base_model
        encoder.config.use_cache = False
        if hasattr(encoder, "gradient_checkpointing_enable"):
            encoder.gradient_checkpointing_enable()
        del causal_lm

        lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            target_modules=["c_attn", "c_proj", "c_fc"],
        )
        self.encoder = get_peft_model(encoder, lora_config)
        self.dropout = nn.Dropout(0.10)
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        output = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = output.last_hidden_state
        last = attention_mask.sum(dim=1) - 1
        pooled = hidden[torch.arange(hidden.size(0), device=hidden.device), last]
        return self.classifier(self.dropout(pooled)).squeeze(-1)


def make_loader(rows, shuffle):
    generator = torch.Generator().manual_seed(SEED)
    return DataLoader(
        CachedTimelineDataset(rows), batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_batch,
        generator=generator if shuffle else None,
    )


def predict(model, loader, device):
    model.eval()
    output_rows, output_prob, output_y = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(ids, mask)
            output_rows.extend(batch["rows"].numpy().tolist())
            output_prob.extend(torch.sigmoid(logits).float().cpu().numpy().tolist())
            output_y.extend(batch["labels"].numpy().tolist())
    return np.asarray(output_rows), np.asarray(output_y, int), np.asarray(output_prob, float)

## 5. Leakage-safe outer-fold fine-tuning

For each outer test fold, 10% of the remaining participants is reserved for validation using a stratified split. Adapter weights are reset from the same pretrained CLMBR checkpoint for every fold. Participants without a usable timeline receive the training-fold prevalence rather than a label-informed value.

Training runs for up to 10 epochs with early stopping after two epochs without sufficient validation-AUPRC improvement; the best validation epoch is restored. Fold predictions and adapters are saved separately. Rerunning this cell reuses a completed fold only when its recorded training configuration matches the current settings.

In [ ]:
device = torch.device("cuda")
all_rows = np.arange(len(cohort))
y_all = cohort.has_lungcancer_3Y.to_numpy(int)

def saved_fold_configuration(fold_id):
    config_path = LORA_DIR / f"fold_{fold_id}_adapter" / "training_config.json"
    if not config_path.exists():
        return None
    try:
        return json.loads(config_path.read_text())
    except (OSError, json.JSONDecodeError):
        return None


def configuration_mismatches(saved):
    if saved is None:
        return {"training_config": ("missing or unreadable", "required")}
    return {
        key: (saved.get(key), expected)
        for key, expected in EXPECTED_TRAINING_CONFIG.items()
        if saved.get(key) != expected
    }


for fold_id in FOLDS_TO_RUN:
    fold_pred_path = LORA_DIR / f"fold_{fold_id}_predictions.parquet"
    fold_dir = LORA_DIR / f"fold_{fold_id}_adapter"
    mismatches = configuration_mismatches(saved_fold_configuration(fold_id))
    force_retrain = fold_id in FORCE_RETRAIN_FOLDS

    if fold_pred_path.exists() and not mismatches and not force_retrain:
        print(f"Fold {fold_id}: compatible saved predictions found; skipping")
        continue

    if force_retrain:
        reason = "listed in FORCE_RETRAIN_FOLDS"
    elif fold_pred_path.exists():
        reason = f"saved configuration mismatch: {mismatches}"
    else:
        reason = "no completed prediction file"
    print(f"Fold {fold_id}: training required ({reason})")

    # Any adapter retraining invalidates fold-specific embeddings and aggregate
    # predictions derived from the previous adapter.
    stale_paths = [
        RUN_DIR / "clmbr_lora_finetuned_embeddings" / f"fold_{fold_id}_embeddings.parquet",
        PRED_DIR / "CLMBR_LoRA_predictions.parquet",
        PRED_DIR / "Demographic_Finetuned_CLMBR_predictions.parquet",
        PRED_DIR / "Demographic_Finetuned_CLMBR_Smoking_predictions.parquet",
        RUN_DIR / "Demographic_Finetuned_CLMBR_slide_metrics.csv",
        RUN_DIR / "Finetuned_CLMBR_slide_metrics.csv",
    ]
    for stale_path in stale_paths:
        if stale_path.exists():
            stale_path.unlink()
            print("Removed stale artifact:", stale_path)

    test_rows = all_rows[cohort.fold.to_numpy() == fold_id]
    train_pool = all_rows[(cohort.fold.to_numpy() != fold_id) & eligible_for_lora]
    train_rows, val_rows = train_test_split(
        train_pool, test_size=VALIDATION_FRACTION, random_state=SEED + fold_id,
        stratify=y_all[train_pool],
    )
    test_model_rows = test_rows[eligible_for_lora[test_rows]]
    train_prevalence = float(y_all[train_rows].mean())

    train_loader = make_loader(train_rows, shuffle=True)
    val_loader = make_loader(val_rows, shuffle=False)
    test_loader = make_loader(test_model_rows, shuffle=False)

    model = CLMBRLoRAClassifier().to(device)
    model.encoder.print_trainable_parameters()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    # T4/V100 use FP16 with gradient scaling; Ampere and newer use BF16.
    scaler = torch.amp.GradScaler("cuda", enabled=not USE_BF16)
    positives = int(y_all[train_rows].sum())
    negatives = len(train_rows) - positives
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(negatives / positives, device=device)
    )

    best_auprc = -np.inf
    best_epoch = None
    epochs_without_improvement = 0
    best_trainable_state = None
    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f"Fold {fold_id} epoch {epoch + 1}"), start=1):
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                logits = model(ids, mask)
                loss = loss_fn(logits, labels) / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            running_loss += float(loss.item()) * GRAD_ACCUM_STEPS
            if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        _, val_y, val_prob = predict(model, val_loader, device)
        val_auprc = average_precision_score(val_y, val_prob)
        print(
            f"Fold {fold_id} epoch {epoch + 1}: "
            f"loss={running_loss / len(train_loader):.5f} "
            f"val_AUROC={roc_auc_score(val_y, val_prob):.4f} "
            f"val_AUPRC={val_auprc:.6f}"
        )
        if val_auprc > best_auprc + EARLY_STOPPING_MIN_DELTA:
            best_auprc = val_auprc
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            best_trainable_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
                if "lora_" in name or name.startswith("classifier.")
            }
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(
                    f"Fold {fold_id}: early stopping after epoch {epoch + 1}; "
                    f"best epoch={best_epoch}, best val_AUPRC={best_auprc:.6f}"
                )
                break

    if best_trainable_state is None:
        raise RuntimeError(f"Fold {fold_id}: no validation checkpoint was saved.")
    model.load_state_dict(best_trainable_state, strict=False)
    print(f"Fold {fold_id}: restoring best epoch {best_epoch}")
    fold_dir.mkdir(parents=True, exist_ok=True)
    model.encoder.save_pretrained(fold_dir)
    torch.save(model.classifier.state_dict(), fold_dir / "classification_head.pt")
    saved_training_config = {
        **EXPECTED_TRAINING_CONFIG,
        "best_epoch": int(best_epoch),
        "validation_auprc": float(best_auprc),
    }
    (fold_dir / "training_config.json").write_text(
        json.dumps(saved_training_config, indent=2)
    )

    fold_prob = np.full(len(test_rows), train_prevalence, dtype=float)
    if len(test_model_rows):
        predicted_rows, test_y_model, predicted_prob = predict(model, test_loader, device)
        row_to_position = {int(row): pos for pos, row in enumerate(test_rows)}
        for row, probability in zip(predicted_rows, predicted_prob):
            fold_prob[row_to_position[int(row)]] = probability

    fold_result = pd.DataFrame({
        "person_id": cohort.iloc[test_rows].person_id.to_numpy(),
        "y_true": y_all[test_rows],
        "prob": fold_prob,
        "model": "CLMBR_LoRA",
        "fold": fold_id,
        "timeline_missing": ~eligible_for_lora[test_rows],
    })
    fold_result.to_parquet(fold_pred_path, index=False)
    print(
        f"Fold {fold_id} test: AUROC={roc_auc_score(fold_result.y_true, fold_result.prob):.4f} "
        f"AUPRC={average_precision_score(fold_result.y_true, fold_result.prob):.6f}"
    )

    del model, optimizer, train_loader, val_loader, test_loader
    gc.collect()
    torch.cuda.empty_cache()

## 6. Assemble and audit OOF predictions

In [ ]:
fold_files = [LORA_DIR / f"fold_{fold}_predictions.parquet" for fold in FOLDS_TO_RUN]
missing_fold_files = [str(path) for path in fold_files if not path.exists()]
if missing_fold_files:
    raise FileNotFoundError(f"Incomplete LoRA run; missing: {missing_fold_files}")

oof = pd.concat([pd.read_parquet(path) for path in fold_files], ignore_index=True)
expected_rows = int(cohort["fold"].isin(FOLDS_TO_RUN).sum())
if len(oof) != expected_rows:
    raise AssertionError(f"Expected {expected_rows} rows for folds {FOLDS_TO_RUN}, found {len(oof)}")
if oof.person_id.duplicated().any():
    raise AssertionError("Duplicate participants in LoRA fold predictions")

expected_subset = cohort.loc[
    cohort["fold"].isin(FOLDS_TO_RUN),
    ["person_id", "has_lungcancer_3Y", "fold"],
]
check = expected_subset.merge(
    oof, on="person_id", how="left", validate="one_to_one", suffixes=("_expected", "_pred")
)
if not np.array_equal(check.has_lungcancer_3Y.to_numpy(), check.y_true.to_numpy()):
    raise AssertionError("Outcome mismatch in LoRA predictions")
if not np.array_equal(check.fold_expected.to_numpy(), check.fold_pred.to_numpy()):
    raise AssertionError("Fold mismatch in LoRA predictions")

print("Completed folds:", sorted(FOLDS_TO_RUN))
print("Participants evaluated:", len(oof))
print("AUROC:", roc_auc_score(oof.y_true, oof.prob))
print("AUPRC:", average_precision_score(oof.y_true, oof.prob))
print("Brier:", brier_score_loss(oof.y_true, oof.prob))
print("Participants using missing-timeline fallback:", int(oof.timeline_missing.sum()))

all_folds = sorted(cohort["fold"].unique().tolist())
if sorted(FOLDS_TO_RUN) == all_folds:
    if len(oof) != len(cohort):
        raise AssertionError(f"Expected {len(cohort)} full OOF rows, found {len(oof)}")
    final_predictions = oof[["person_id", "y_true", "prob", "model"]].copy()
    final_path = PRED_DIR / "CLMBR_LoRA_predictions.parquet"
    final_predictions.to_parquet(final_path, index=False)
    print("Saved full OOF predictions:", final_path)
else:
    print(
        "Smoke test complete. Set FOLDS_TO_RUN =", all_folds,
        "and rerun the training cell; completed folds will be skipped."
    )

## 7. Demographic + fine-tuned CLMBR embeddings

This downstream experiment reuses each completed outer-fold LoRA adapter. For fold `k`, that adapter was trained without fold `k`; it is used to extract representations for the outer-training participants and the held-out participants, after which XGBoost is trained on demographics plus those fold-specific representations and evaluated on fold `k`. CLMBR is not fine-tuned again. The embedding caches and predictions are controlled participant-level artifacts and must remain inside the controlled workspace.

In [ ]:
from peft import PeftModel
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

FINETUNED_EMBED_DIR = RUN_DIR / "clmbr_lora_finetuned_embeddings"
FINETUNED_EMBED_DIR.mkdir(parents=True, exist_ok=True)
EMBED_BATCH_SIZE = 64

all_folds = sorted(cohort["fold"].unique().tolist())
if sorted(FOLDS_TO_RUN) != all_folds:
    raise ValueError(
        f"Set FOLDS_TO_RUN={all_folds} before building the full combined model."
    )
missing_adapters = [
    str(LORA_DIR / f"fold_{fold}_adapter")
    for fold in all_folds
    if not (LORA_DIR / f"fold_{fold}_adapter" / "adapter_config.json").exists()
]
if missing_adapters:
    raise FileNotFoundError(f"Missing completed fold adapters: {missing_adapters}")

demographic_columns = [
    "age_at_survey_int",
    "sex_at_birth",
    "race_ethnicity",
    "education_group",
]
missing_demographic_columns = [
    column for column in demographic_columns if column not in cohort.columns
]
if missing_demographic_columns:
    raise KeyError(f"Missing demographic columns: {missing_demographic_columns}")

demographic = cohort[demographic_columns].copy()
demographic["age_at_survey_int"] = pd.to_numeric(
    demographic["age_at_survey_int"], errors="coerce"
)
demographic = pd.get_dummies(
    demographic,
    columns=["sex_at_birth", "race_ethnicity", "education_group"],
    dummy_na=True,
).apply(pd.to_numeric, errors="coerce")


def extract_fold_tuned_embeddings(fold_id, rows):
    adapter_dir = LORA_DIR / f"fold_{fold_id}_adapter"
    causal_lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    base_encoder = causal_lm.base_model
    base_encoder.config.use_cache = False
    del causal_lm
    encoder = PeftModel.from_pretrained(
        base_encoder, str(adapter_dir), is_trainable=False
    ).to(device).eval()

    loader = DataLoader(
        CachedTimelineDataset(rows),
        batch_size=EMBED_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        collate_fn=collate_batch,
    )
    output_rows, vectors = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Fold {fold_id} tuned embeddings"):
            ids = batch["input_ids"].to(device, non_blocking=True)
            mask = batch["attention_mask"].to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                output = encoder(input_ids=ids, attention_mask=mask)
            hidden = output.last_hidden_state
            last = mask.sum(dim=1) - 1
            pooled = hidden[
                torch.arange(hidden.size(0), device=hidden.device), last
            ]
            output_rows.extend(batch["rows"].numpy().tolist())
            vectors.append(pooled.float().cpu().numpy())

    matrix = np.concatenate(vectors, axis=0)
    columns = [f"clmbr_ft_{index:04d}" for index in range(matrix.shape[1])]
    result = pd.DataFrame(matrix, columns=columns)
    result.insert(0, "person_id", cohort.iloc[output_rows].person_id.to_numpy())

    del encoder, base_encoder, loader, matrix, vectors
    gc.collect()
    torch.cuda.empty_cache()
    return result


combined_oof = np.zeros(len(cohort), dtype=float)
combined_fold_aurocs = []
combined_fold_auprcs = []
y = cohort["has_lungcancer_3Y"].astype(int)
eligible_rows = np.flatnonzero(eligible_for_lora)

for fold_id in all_folds:
    embedding_path = FINETUNED_EMBED_DIR / f"fold_{fold_id}_embeddings.parquet"
    if embedding_path.exists():
        fold_embeddings = pd.read_parquet(embedding_path)
        print(f"Fold {fold_id}: using cached fine-tuned embeddings")
    else:
        fold_embeddings = extract_fold_tuned_embeddings(fold_id, eligible_rows)
        fold_embeddings.to_parquet(embedding_path, index=False)
        print("Saved controlled embedding cache:", embedding_path)

    if fold_embeddings["person_id"].duplicated().any():
        raise AssertionError(f"Fold {fold_id}: duplicate embedding person IDs")
    embedding_columns = [
        column for column in fold_embeddings.columns if column != "person_id"
    ]
    aligned = cohort[["person_id"]].merge(
        fold_embeddings, on="person_id", how="left", validate="one_to_one"
    )
    embedding_frame = aligned[embedding_columns].apply(pd.to_numeric, errors="coerce")
    embedding_frame["embedding_missing"] = (
        embedding_frame.isna().all(axis=1).astype(int)
    )
    X_fold = pd.concat([demographic, embedding_frame], axis=1)

    test = cohort["fold"].eq(fold_id).to_numpy()
    train = ~test
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_fold.loc[train])
    X_test = imputer.transform(X_fold.loc[test])

    downstream = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
    )
    downstream.fit(X_train, y.loc[train])
    combined_oof[test] = downstream.predict_proba(X_test)[:, 1]

    fold_auc = roc_auc_score(y.loc[test], combined_oof[test])
    fold_ap = average_precision_score(y.loc[test], combined_oof[test])
    combined_fold_aurocs.append(fold_auc)
    combined_fold_auprcs.append(fold_ap)
    print(f"Fold {fold_id + 1}: AUROC={fold_auc:.4f}; AUPRC={fold_ap:.4f}")

    del fold_embeddings, aligned, embedding_frame, X_fold
    del X_train, X_test, downstream, imputer
    gc.collect()

combined_pred = pd.DataFrame({
    "person_id": cohort["person_id"],
    "y_true": y,
    "prob": combined_oof,
    "model": "Demographic_Finetuned_CLMBR",
})
combined_path = PRED_DIR / "Demographic_Finetuned_CLMBR_predictions.parquet"
combined_pred.to_parquet(combined_path, index=False)

print("Saved full OOF predictions:", combined_path)
print("OOF AUROC:", roc_auc_score(y, combined_oof))
print("OOF AUPRC:", average_precision_score(y, combined_oof))
print("OOF Brier:", brier_score_loss(y, combined_oof))
print(f"Fold AUROC: {np.mean(combined_fold_aurocs):.3f} ± {np.std(combined_fold_aurocs):.3f}")
print(f"Fold AUPRC: {np.mean(combined_fold_auprcs):.3f} ± {np.std(combined_fold_auprcs):.3f}")


def top_capacity_metrics(predictions, fractions=(0.05, 0.10, 0.20)):
    ranked = predictions.sort_values(
        ["prob", "person_id"], ascending=[False, True]
    ).reset_index(drop=True)
    total_cases = int(ranked["y_true"].sum())
    rows = []
    for fraction in fractions:
        selected_n = int(np.floor(len(ranked) * fraction))
        captured = int(ranked.loc[: selected_n - 1, "y_true"].sum())
        rows.append({
            "Capacity": f"Top {int(fraction * 100)}%",
            "Sensitivity": captured / total_cases,
            "PPV": captured / selected_n,
        })
    return pd.DataFrame(rows).set_index("Capacity")


def slide_metric_row(label, predictions):
    capacity = top_capacity_metrics(predictions)
    return {
        "Model": label,
        "AUROC": roc_auc_score(predictions["y_true"], predictions["prob"]),
        "AUPRC": average_precision_score(predictions["y_true"], predictions["prob"]),
        "Top 5% Sens./PPV": (
            f"{capacity.loc['Top 5%', 'Sensitivity']:.1%} / "
            f"{capacity.loc['Top 5%', 'PPV']:.1%}"
        ),
        "Top 10% Sens./PPV": (
            f"{capacity.loc['Top 10%', 'Sensitivity']:.1%} / "
            f"{capacity.loc['Top 10%', 'PPV']:.1%}"
        ),
        "Top 20% Sens./PPV": (
            f"{capacity.loc['Top 20%', 'Sensitivity']:.1%} / "
            f"{capacity.loc['Top 20%', 'PPV']:.1%}"
        ),
    }


comparison_summary = pd.DataFrame([
    slide_metric_row("Fine-tuned CLMBR", oof),
    slide_metric_row("Demographic + fine-tuned CLMBR", combined_pred),
]).set_index("Model")
display(comparison_summary.style.format({"AUROC": "{:.3f}", "AUPRC": "{:.3f}"}))
comparison_summary.to_csv(
    RUN_DIR / "Demographic_Finetuned_CLMBR_slide_metrics.csv"
)


## 7B. Demographic + fine-tuned CLMBR + smoking

Adds smoking status, smoking duration, usual cigarettes per day, derived pack-years, and years since quitting to the fold-specific fine-tuned CLMBR representations and demographics. This cell reuses the cached fold-specific tuned embeddings; it does not fine-tune CLMBR again.


In [ ]:
def numeric_cohort_column(frame, name):
    if name not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype=float, name=name)
    return pd.to_numeric(frame[name], errors="coerce")

if "smoking_status" not in cohort.columns:
    raise RuntimeError("smoking_status is required for the smoking-augmented model")

smoking_status = (
    cohort["smoking_status"].astype("string").str.strip().str.lower()
)
smoking_years = numeric_cohort_column(cohort, "Smoking: Number Of Years")
average_daily = numeric_cohort_column(
    cohort, "Smoking: Average Daily Cigarette Number"
)
current_daily = numeric_cohort_column(
    cohort, "Smoking: Current Daily Cigarette Number"
)
quit_age = numeric_cohort_column(
    cohort, "Attempt Quit Smoking: Completely Quit Age"
)
age_numeric = numeric_cohort_column(cohort, "age_at_survey_int")

# Match notebook 05A: prefer usual/lifetime smoking intensity, use current
# daily cigarettes only as a fallback for current smokers, and treat zero
# cigarettes/day as missing.
cigs_per_day = average_daily.mask(average_daily <= 0)
current_fallback = smoking_status.eq("current") & cigs_per_day.isna()
valid_current_daily = current_daily.mask(current_daily <= 0)
cigs_per_day.loc[current_fallback] = valid_current_daily.loc[current_fallback]

years_since_quit = pd.Series(np.nan, index=cohort.index, dtype=float)
years_since_quit.loc[smoking_status.eq("current")] = 0.0
former = smoking_status.eq("former")
years_since_quit.loc[former] = age_numeric.loc[former] - quit_age.loc[former]
years_since_quit = years_since_quit.mask(years_since_quit < 0)

smoking = pd.DataFrame({
    "smoking_status": smoking_status,
    "smoking_years": smoking_years,
    "cigs_per_day": cigs_per_day,
    "pack_years": (cigs_per_day / 20.0) * smoking_years,
    "years_since_quit": years_since_quit,
})
smoking = pd.get_dummies(
    smoking, columns=["smoking_status"], dummy_na=True
).apply(pd.to_numeric, errors="coerce")
demographic_smoking = pd.concat([demographic, smoking], axis=1)
demographic_smoking.columns = demographic_smoking.columns.astype(str)
demographic_smoking = demographic_smoking.loc[
    :, ~demographic_smoking.columns.duplicated()
]

print("Smoking features:", list(smoking.columns))
print("Missing usable cigarettes/day:", int(cigs_per_day.isna().sum()))
print("Missing smoking duration:", int(smoking_years.isna().sum()))
print("Former smokers missing quit age:", int((former & quit_age.isna()).sum()))

combined_smoking_oof = np.zeros(len(cohort), dtype=float)
combined_smoking_fold_aurocs = []
combined_smoking_fold_auprcs = []

for fold_id in all_folds:
    embedding_path = FINETUNED_EMBED_DIR / f"fold_{fold_id}_embeddings.parquet"
    if not embedding_path.exists():
        raise FileNotFoundError(
            f"Missing {embedding_path}; run section 7 first."
        )
    fold_embeddings = pd.read_parquet(embedding_path)
    if fold_embeddings["person_id"].duplicated().any():
        raise AssertionError(f"Fold {fold_id}: duplicate embedding person IDs")

    embedding_columns = [
        column for column in fold_embeddings.columns if column != "person_id"
    ]
    aligned = cohort[["person_id"]].merge(
        fold_embeddings, on="person_id", how="left", validate="one_to_one"
    )
    embedding_frame = aligned[embedding_columns].apply(
        pd.to_numeric, errors="coerce"
    )
    embedding_frame["embedding_missing"] = (
        embedding_frame.isna().all(axis=1).astype(int)
    )
    X_fold = pd.concat([demographic_smoking, embedding_frame], axis=1)
    X_fold.columns = X_fold.columns.astype(str)
    X_fold = X_fold.loc[:, ~X_fold.columns.duplicated()]

    test = cohort["fold"].eq(fold_id).to_numpy()
    train = ~test
    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_fold.loc[train])
    X_test = imputer.transform(X_fold.loc[test])

    downstream = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=SEED,
    )
    downstream.fit(X_train, y.loc[train])
    combined_smoking_oof[test] = downstream.predict_proba(X_test)[:, 1]

    fold_auc = roc_auc_score(y.loc[test], combined_smoking_oof[test])
    fold_ap = average_precision_score(y.loc[test], combined_smoking_oof[test])
    combined_smoking_fold_aurocs.append(fold_auc)
    combined_smoking_fold_auprcs.append(fold_ap)
    print(f"Fold {fold_id + 1}: AUROC={fold_auc:.4f}; AUPRC={fold_ap:.4f}")

    del fold_embeddings, aligned, embedding_frame, X_fold
    del X_train, X_test, downstream, imputer
    gc.collect()

combined_smoking_pred = pd.DataFrame({
    "person_id": cohort["person_id"],
    "y_true": y,
    "prob": combined_smoking_oof,
    "model": "Demographic_Finetuned_CLMBR_Smoking",
})
combined_smoking_path = (
    PRED_DIR / "Demographic_Finetuned_CLMBR_Smoking_predictions.parquet"
)
combined_smoking_pred.to_parquet(combined_smoking_path, index=False)

print("Saved full OOF predictions:", combined_smoking_path)
print("OOF AUROC:", roc_auc_score(y, combined_smoking_oof))
print("OOF AUPRC:", average_precision_score(y, combined_smoking_oof))
print("OOF Brier:", brier_score_loss(y, combined_smoking_oof))
print(
    f"Fold AUROC: {np.mean(combined_smoking_fold_aurocs):.3f} ± "
    f"{np.std(combined_smoking_fold_aurocs):.3f}"
)
print(
    f"Fold AUPRC: {np.mean(combined_smoking_fold_auprcs):.3f} ± "
    f"{np.std(combined_smoking_fold_auprcs):.3f}"
)

comparison_summary_with_smoking = pd.concat([
    comparison_summary,
    pd.DataFrame([
        slide_metric_row(
            "Demographic + fine-tuned CLMBR + Smoking",
            combined_smoking_pred,
        )
    ]).set_index("Model"),
])
display(comparison_summary_with_smoking.style.format({
    "AUROC": "{:.3f}", "AUPRC": "{:.3f}"
}))
comparison_summary_with_smoking.to_csv(
    RUN_DIR / "Finetuned_CLMBR_slide_metrics.csv"
)


## Full-cohort comparison

This diagnostic compares the five-fold out-of-fold CLMBR-only LoRA predictions with the demographic + fine-tuned-embedding and demographic + fine-tuned-embedding + smoking predictions on the same participants.


In [ ]:
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

pre_path = PRED_DIR / "CLMBR_LoRA_predictions.parquet"
post_path = PRED_DIR / "Demographic_Finetuned_CLMBR_predictions.parquet"
smoking_path = (
    PRED_DIR / "Demographic_Finetuned_CLMBR_Smoking_predictions.parquet"
)

if not post_path.exists():
    raise FileNotFoundError(f"Missing {post_path}; run the combined-model cell first.")
if not smoking_path.exists():
    raise FileNotFoundError(f"Missing {smoking_path}; run section 7B first.")
if not pre_path.exists():
    raise FileNotFoundError(
        f"Missing five-fold LoRA benchmark {pre_path}. Complete all five "
        "LoRA folds and assemble their OOF predictions first."
    )

pre = pd.read_parquet(pre_path)[["person_id", "y_true", "prob"]].rename(
    columns={"y_true": "y_pre", "prob": "prob_pre"}
)
post = pd.read_parquet(post_path)[["person_id", "y_true", "prob"]].rename(
    columns={"y_true": "y_post", "prob": "prob_post"}
)
smoking_post = pd.read_parquet(
    smoking_path
)[["person_id", "y_true", "prob"]].rename(
    columns={"y_true": "y_smoking", "prob": "prob_smoking"}
)
comparison = pre.merge(
    post, on="person_id", how="inner", validate="one_to_one"
).merge(
    smoking_post, on="person_id", how="inner", validate="one_to_one"
)

expected_ids = set(cohort["person_id"])
if set(comparison.person_id) != expected_ids:
    raise AssertionError(
        "Comparison does not cover the complete cohort: "
        f"expected {len(expected_ids)}, found {len(comparison)}."
    )
if not np.array_equal(comparison.y_pre.to_numpy(), comparison.y_post.to_numpy()):
    raise AssertionError("Pre/post outcomes do not match.")
if not np.array_equal(comparison.y_post.to_numpy(), comparison.y_smoking.to_numpy()):
    raise AssertionError("Smoking-model outcomes do not match.")

def metric_row(label, y, probability):
    return {
        "model": label,
        "n": len(y),
        "cases": int(np.sum(y)),
        "prevalence": float(np.mean(y)),
        "AUROC": roc_auc_score(y, probability),
        "AUPRC": average_precision_score(y, probability),
        "Brier": brier_score_loss(y, probability),
    }

y_fold = comparison.y_post.to_numpy()
results = pd.DataFrame([
    metric_row("Fine-tuned CLMBR only", y_fold, comparison.prob_pre),
    metric_row("Demographic + fine-tuned CLMBR embeddings", y_fold, comparison.prob_post),
    metric_row(
        "Demographic + fine-tuned CLMBR embeddings + smoking",
        y_fold,
        comparison.prob_smoking,
    ),
])
display(results.style.format({
    "prevalence": "{:.3%}", "AUROC": "{:.4f}",
    "AUPRC": "{:.6f}", "Brier": "{:.6f}",
}))

delta = results.set_index("model").loc["Demographic + fine-tuned CLMBR embeddings", ["AUROC", "AUPRC", "Brier"]] - results.set_index("model").loc["Fine-tuned CLMBR only", ["AUROC", "AUPRC", "Brier"]]
print("Full-cohort deltas (demographic + embeddings minus CLMBR-only LoRA):")
print(f"  AUROC: {delta.AUROC:+.4f}")
print(f"  AUPRC: {delta.AUPRC:+.6f}")
print(f"  Brier: {delta.Brier:+.6f}  (negative is better)")
smoking_delta = results.set_index("model").loc[
    "Demographic + fine-tuned CLMBR embeddings + smoking",
    ["AUROC", "AUPRC", "Brier"],
] - results.set_index("model").loc[
    "Demographic + fine-tuned CLMBR embeddings",
    ["AUROC", "AUPRC", "Brier"],
]
print("Incremental deltas from adding smoking to demographics + tuned embeddings:")
print(f"  AUROC: {smoking_delta.AUROC:+.4f}")
print(f"  AUPRC: {smoking_delta.AUPRC:+.6f}")
print(f"  Brier: {smoking_delta.Brier:+.6f}  (negative is better)")
print("Participants compared:", len(comparison))


## 8. Save controlled artifacts to the Workbench bucket

Adapters and predictions may contain information learned from controlled participant data. Keep them in the controlled workspace bucket and do not commit them to Git.

In [ ]:
import subprocess

if SYNTHETIC_MODE:
    print("Synthetic mode: skipping upload of test artifacts to the Workbench bucket.")
else:
    bucket = resolve_workspace_bucket()

    subprocess.run([
        "wb", "gcloud", "storage", "cp", "--recursive",
        str(LORA_DIR), f"{bucket}/clmbr/lora/",
    ], check=True)
    prediction_path = PRED_DIR / "CLMBR_LoRA_predictions.parquet"
    if prediction_path.exists():
        subprocess.run([
            "wb", "gcloud", "storage", "cp", str(prediction_path),
            f"{bucket}/clmbr/CLMBR_LoRA_predictions.parquet",
        ], check=True)
    else:
        print("Full five-fold OOF prediction file is not available yet; uploading adapters only.")
    combined_prediction_path = (
        PRED_DIR / "Demographic_Finetuned_CLMBR_predictions.parquet"
    )
    if combined_prediction_path.exists():
        subprocess.run([
            "wb", "gcloud", "storage", "cp", str(combined_prediction_path),
            f"{bucket}/clmbr/Demographic_Finetuned_CLMBR_predictions.parquet",
        ], check=True)
    print("Uploaded available controlled LoRA artifacts to", f"{bucket}/clmbr/")


## Reporting note

Report `CLMBR_LoRA` as **task-specific LoRA fine-tuning of CLMBR**. Report `Demographic_Finetuned_CLMBR` as **demographics plus fold-specific LoRA-tuned CLMBR embeddings with a downstream XGBoost classifier**. Report `Demographic_Finetuned_CLMBR_Smoking` separately as the same downstream model augmented with smoking status, duration, intensity, pack-years, and years since quitting. LoRA fine-tuning runs for up to 10 epochs with early stopping patience 2 and restores the epoch with the best validation AUPRC. All estimates use the same five held-out outer folds. Notebook 06 can compare the saved OOF prediction files with the demographic + frozen-CLMBR, clinical, and Phecode models.